# Integrated summary visualization

This notebook creates one experiment-wide summary figure that combines outputs from the existing analysis notebooks:

- **Psychophysics output**: trial order, stiffness condition, and correct/incorrect comparison response.
- **XY probing / kinematic output**: time-binned hand/object displacement from center across each trial.

The figure encodes four signals at once:

1. **X-axis** = continuous experiment time across repetitions/trials (`global_trial_order + within-trial time`).
2. **Y-axis** = stacked X/Y panels: X-relative and Y-relative location, or X/Y velocity in the export script.
3. **Thin arrows** = local hand movement direction estimated from a short before/after time window in XY space.
4. **Arrow color** = local object stiffness level.
5. **Background** = behavioral outcome per trial: blue = correct, red = incorrect.

## Configuration

Set `SUBJECT_ID` to the participant/session you want to visualize. Most experiment subjects have 268 trials; probing subjects may have 256.

The notebook reads only the columns needed from the large trajectory CSV and filters it in chunks so it can run without loading the full file into memory.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Participant/session to visualize. Change this to any available subject_id.
SUBJECT_ID = "L_E_1"

# Existing outputs created by the analysis notebooks.
PSYCHOPHYSICS_TRIALS_CSV = Path("analysis/psychophysics_analysis/results/combined_success_trials.csv")
XY_TRAJECTORY_BINS_CSV = Path("analysis/probing_analysis/results/xy_probing_skin_stretch_trajectory_bins.csv")

# Direction arrows: local before/after window within each trial, measured in binned samples.
MOTION_WINDOW_BINS = 3
MAX_ARROWS = 750

# Arrow glyph size in plot data units. These are visual glyph lengths, not physical velocities.
ARROW_LENGTH_X_TRIALS = 0.22
ARROW_LENGTH_Y_DISTANCE = 7.0

# Output files.
OUTPUT_DIR = Path("analysis/summery")
OUTPUT_BASENAME = f"integrated_summary_visualization_{SUBJECT_ID}"
OUTPUT_PNG = OUTPUT_DIR / f"{OUTPUT_BASENAME}.png"
OUTPUT_SVG = OUTPUT_DIR / f"{OUTPUT_BASENAME}.svg"

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    """Return repo root whether the notebook is run from the repo root or from summery/."""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "analysis").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root containing analysis/ and pyproject.toml")

REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PSYCHOPHYSICS_TRIALS_CSV = REPO_ROOT / PSYCHOPHYSICS_TRIALS_CSV
XY_TRAJECTORY_BINS_CSV = REPO_ROOT / XY_TRAJECTORY_BINS_CSV
OUTPUT_PNG = REPO_ROOT / OUTPUT_PNG
OUTPUT_SVG = REPO_ROOT / OUTPUT_SVG

print("Repo root:", REPO_ROOT)
print("Psychophysics trials:", PSYCHOPHYSICS_TRIALS_CSV)
print("XY trajectory bins:", XY_TRAJECTORY_BINS_CSV)
print("Output PNG:", OUTPUT_PNG)

## Load trial-level behavior and stiffness metadata

`combined_success_trials.csv` provides the response outcome for each trial. This is used for the blue/red background segmentation and to validate that the selected subject spans the expected trial count.

In [ ]:
def load_trial_table(path: Path, subject_id: str) -> pd.DataFrame:
    trial_cols = [
        "subject_id",
        "global_trial_order",
        "trial_index_raw",
        "timestamp",
        "elapsed_seconds",
        "comparison_value",
        "standard_value",
        "signed_stiffness_delta",
        "correct_response",
        "incorrect_response",
        "success_label",
        "finger_condition",
        "reaction_time",
    ]
    available = pd.read_csv(path, nrows=0).columns.tolist()
    usecols = [c for c in trial_cols if c in available]
    trials = pd.read_csv(path, usecols=usecols)
    trials = trials.loc[trials["subject_id"].astype(str) == subject_id].copy()
    if trials.empty:
        available_subjects = pd.read_csv(path, usecols=["subject_id"])["subject_id"].dropna().astype(str).unique()
        raise ValueError(f"No rows for SUBJECT_ID={subject_id!r}. Available examples: {available_subjects[:20]}")
    trials["global_trial_order"] = pd.to_numeric(trials["global_trial_order"], errors="coerce")
    trials["correct_response"] = pd.to_numeric(trials["correct_response"], errors="coerce")
    trials = trials.sort_values("global_trial_order").reset_index(drop=True)
    return trials

trials = load_trial_table(PSYCHOPHYSICS_TRIALS_CSV, SUBJECT_ID)
print(trials[["subject_id", "global_trial_order", "comparison_value", "standard_value", "correct_response", "success_label"]].head())
print(f"{SUBJECT_ID}: {len(trials)} trial rows, order {trials['global_trial_order'].min():.0f}-{trials['global_trial_order'].max():.0f}")

## Load time-binned XY/distance data

This uses the XY probing output because it contains one row per within-trial time bin with:

- `object_dx_from_center_px`, `object_dy_from_center_px` for direction estimation,
- `center_distance_px` for the Y-axis,
- `skin_stretch_gain_mm_per_m_or_condition` / `skin_stretch_gain_mm_per_m` for local stiffness color.

In [ ]:
def load_subject_trajectory(path: Path, subject_id: str, chunksize: int = 300_000) -> pd.DataFrame:
    wanted = [
        "subject_id",
        "global_trial_order",
        "trial_index_raw",
        "trajectory_time_bin",
        "time_fraction",
        "object_dx_from_center_px",
        "object_dy_from_center_px",
        "center_distance_px",
        "skin_stretch_gain_mm_per_m_or_condition",
        "skin_stretch_gain_mm_per_m",
        "comparison_value",
        "standard_value",
        "correct_response",
        "reaction_time",
        "finger_condition",
        "tracking_file",
    ]
    available = pd.read_csv(path, nrows=0).columns.tolist()
    usecols = [c for c in wanted if c in available]
    pieces = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize):
        chunk = chunk.loc[chunk["subject_id"].astype(str) == subject_id].copy()
        if not chunk.empty:
            pieces.append(chunk)
    if not pieces:
        raise ValueError(f"No trajectory rows found for SUBJECT_ID={subject_id!r}")
    traj = pd.concat(pieces, ignore_index=True)
    numeric_cols = [c for c in traj.columns if c not in {"subject_id", "finger_condition", "tracking_file"}]
    for col in numeric_cols:
        traj[col] = pd.to_numeric(traj[col], errors="coerce")
    traj = traj.sort_values(["global_trial_order", "trajectory_time_bin", "time_fraction"]).reset_index(drop=True)
    return traj

trajectory = load_subject_trajectory(XY_TRAJECTORY_BINS_CSV, SUBJECT_ID)
print(trajectory.head())
print(f"Loaded {len(trajectory):,} binned trajectory rows for {SUBJECT_ID}")
print(f"Trials represented: {trajectory['global_trial_order'].nunique()} unique trials")

## Prepare continuous time, distance, local motion vectors, and plot colors

The local motion vector is estimated as:

$$
\Delta x = x(t + w) - x(t - w), \quad \Delta y = y(t + w) - y(t - w)
$$

where `w = MOTION_WINDOW_BINS`. The vector is computed **within each trial** so arrows do not bridge across trial boundaries.

In [ ]:
def prepare_plot_data(trajectory: pd.DataFrame, trials: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    data = trajectory.copy()

    # Merge trial-level behavioral outcome from psychophysics output when available.
    trial_cols = ["global_trial_order", "correct_response", "success_label"]
    trial_info = trials[trial_cols].drop_duplicates("global_trial_order").copy()
    data = data.drop(columns=[c for c in ["correct_response", "success_label"] if c in data.columns], errors="ignore")
    data = data.merge(trial_info, on="global_trial_order", how="left")

    # Continuous experiment axis: trial 1 spans x=[0,1), trial 268 spans x=[267,268).
    data["continuous_trial_time"] = (data["global_trial_order"] - 1) + data["time_fraction"].clip(0, 1)
    data["distance_from_center_px"] = data["center_distance_px"].abs()

    # Local stiffness: prefer the local trajectory-bin value, then fallback to the condition value.
    if "skin_stretch_gain_mm_per_m_or_condition" in data:
        data["local_stiffness"] = data["skin_stretch_gain_mm_per_m_or_condition"]
    elif "skin_stretch_gain_mm_per_m" in data:
        data["local_stiffness"] = data["skin_stretch_gain_mm_per_m"]
    else:
        data["local_stiffness"] = data.get("comparison_value", np.nan)
    data["local_stiffness"] = pd.to_numeric(data["local_stiffness"], errors="coerce")

    # Central-difference motion vector within each trial.
    group = data.groupby("global_trial_order", sort=False)
    x_prev = group["object_dx_from_center_px"].shift(MOTION_WINDOW_BINS)
    x_next = group["object_dx_from_center_px"].shift(-MOTION_WINDOW_BINS)
    y_prev = group["object_dy_from_center_px"].shift(MOTION_WINDOW_BINS)
    y_next = group["object_dy_from_center_px"].shift(-MOTION_WINDOW_BINS)
    data["motion_dx_px"] = x_next - x_prev
    data["motion_dy_px"] = y_next - y_prev
    data["motion_norm_px"] = np.hypot(data["motion_dx_px"], data["motion_dy_px"])
    data["motion_angle_rad"] = np.arctan2(data["motion_dy_px"], data["motion_dx_px"])

    trial_segments = trials[["global_trial_order", "correct_response"]].drop_duplicates("global_trial_order").copy()
    trial_segments["x_start"] = trial_segments["global_trial_order"] - 1
    trial_segments["x_end"] = trial_segments["global_trial_order"]
    trial_segments["is_correct"] = trial_segments["correct_response"].astype(float) == 1.0
    return data, trial_segments

plot_data, trial_segments = prepare_plot_data(trajectory, trials)
plot_data[["continuous_trial_time", "distance_from_center_px", "local_stiffness", "motion_dx_px", "motion_dy_px", "correct_response"]].head()

## Build the integrated figure

Interpretation notes:

- The export script now writes stacked 2x1 X/Y panels for relative location and X/Y velocity.
- Arrow **orientation** comes from local XY movement direction, but the arrow glyph is drawn on the time-distance plot; it is a direction marker, not a velocity vector in the plot's x/y units.
- Arrow color is continuous stiffness.
- Background bands are trial-level response outcomes.

In [ ]:
def select_arrow_rows(data: pd.DataFrame, max_arrows: int) -> pd.DataFrame:
    arrows = data.dropna(
        subset=[
            "continuous_trial_time",
            "distance_from_center_px",
            "motion_angle_rad",
            "motion_norm_px",
            "local_stiffness",
        ]
    ).copy()
    arrows = arrows.loc[arrows["motion_norm_px"] > 0]
    if arrows.empty:
        raise ValueError("No valid local motion vectors for arrow overlay.")
    stride = max(1, math.ceil(len(arrows) / max_arrows))
    arrows = arrows.iloc[::stride].copy()
    arrows["arrow_u"] = np.cos(arrows["motion_angle_rad"]) * ARROW_LENGTH_X_TRIALS
    arrows["arrow_v"] = np.sin(arrows["motion_angle_rad"]) * ARROW_LENGTH_Y_DISTANCE
    return arrows


def plot_integrated_summary(data: pd.DataFrame, trial_segments: pd.DataFrame, subject_id: str):
    clean = data.dropna(subset=["continuous_trial_time", "distance_from_center_px"]).copy()
    arrows = select_arrow_rows(clean, MAX_ARROWS)

    fig, ax = plt.subplots(figsize=(24, 7), constrained_layout=True)

    # Background: correct/incorrect by trial.
    correct_color = "#7db7ff"   # blue
    incorrect_color = "#ff8f8f" # red
    for row in trial_segments.itertuples(index=False):
        ax.axvspan(
            row.x_start,
            row.x_end,
            color=correct_color if row.is_correct else incorrect_color,
            alpha=0.24,
            lw=0,
            zorder=0,
        )

    # Distance trace.
    ax.plot(
        clean["continuous_trial_time"],
        clean["distance_from_center_px"],
        color="black",
        lw=0.55,
        alpha=0.70,
        zorder=2,
        label="relative location / velocity component",
    )

    # Optional light rolling trend to make the dense trace easier to read.
    trend = clean[["continuous_trial_time", "distance_from_center_px"]].copy()
    trend["distance_trend"] = trend["distance_from_center_px"].rolling(121, center=True, min_periods=10).median()
    ax.plot(
        trend["continuous_trial_time"],
        trend["distance_trend"],
        color="#111111",
        lw=1.5,
        alpha=0.85,
        zorder=3,
        label="rolling median distance",
    )

    # Arrow color scale = stiffness.
    stiffness_values = arrows["local_stiffness"].astype(float)
    norm = mpl.colors.Normalize(vmin=float(np.nanmin(stiffness_values)), vmax=float(np.nanmax(stiffness_values)))
    cmap = mpl.cm.viridis
    q = ax.quiver(
        arrows["continuous_trial_time"],
        arrows["distance_from_center_px"],
        arrows["arrow_u"],
        arrows["arrow_v"],
        stiffness_values,
        cmap=cmap,
        norm=norm,
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.0014,
        headwidth=3.0,
        headlength=4.0,
        headaxislength=3.5,
        alpha=0.95,
        zorder=5,
    )
    cbar = fig.colorbar(q, ax=ax, pad=0.01)
    cbar.set_label("Object stiffness / skin-stretch gain")

    # Legends and labels.
    outcome_legend = [
        Patch(facecolor=correct_color, alpha=0.35, label="correct comparison response"),
        Patch(facecolor=incorrect_color, alpha=0.35, label="incorrect comparison response"),
    ]
    line_legend = ax.legend(loc="upper right", frameon=True)
    ax.add_artist(line_legend)
    ax.legend(handles=outcome_legend, loc="upper left", frameon=True)

    max_trial = int(np.nanmax(trial_segments["global_trial_order"]))
    ax.set_xlim(0, max_trial)
    ax.set_xlabel("Experiment time (continuous trial index; 0-268 for 268 repetitions)")
    ax.set_ylabel("Relative location / velocity component")
    ax.set_title(
        f"Integrated distance, movement direction, stiffness, and behavioral outcome - {subject_id}\n"
        "Background: blue=correct, red=incorrect; arrows: local XY direction, colored by stiffness"
    )
    ax.grid(True, axis="y", alpha=0.22)

    # Major ticks every 20 trials, plus final trial endpoint.
    major_ticks = list(range(0, max_trial + 1, 20))
    if max_trial not in major_ticks:
        major_ticks.append(max_trial)
    ax.set_xticks(major_ticks)

    return fig, ax, arrows

fig, ax, arrows_used = plot_integrated_summary(plot_data, trial_segments, SUBJECT_ID)
print(f"Arrows drawn: {len(arrows_used):,}")
print(f"Correct trials: {int(trial_segments['is_correct'].sum())}; incorrect trials: {int((~trial_segments['is_correct']).sum())}")
plt.show()

## Save figure

The figure is saved to the `summery/` folder as both PNG and SVG for manuscript/presentation use.

In [ ]:
fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_SVG, bbox_inches="tight")
print("Saved:")
print(" -", OUTPUT_PNG)
print(" -", OUTPUT_SVG)

## Quick diagnostics

Use this cell to confirm that the selected subject has the expected experiment length and that stiffness/outcome values are available.

In [ ]:
diagnostics = {
    "subject_id": SUBJECT_ID,
    "trial_rows": int(len(trials)),
    "trial_order_min": int(trials["global_trial_order"].min()),
    "trial_order_max": int(trials["global_trial_order"].max()),
    "trajectory_rows": int(len(trajectory)),
    "trajectory_trials": int(trajectory["global_trial_order"].nunique()),
    "correct_trials": int(trial_segments["is_correct"].sum()),
    "incorrect_trials": int((~trial_segments["is_correct"]).sum()),
    "stiffness_values": sorted(float(x) for x in plot_data["local_stiffness"].dropna().unique()),
}
diagnostics

## Batch export: finger-annotated summary PNGs

This corrected export saves **one full experiment graph per subset**. Finger appearances are shown as letters above the timeline with down-facing brackets.

Script outputs are saved under `analysis/summery/results/distances/` and `analysis/summery/results/velocity/` for all participants, cohorts `N_E/N_P/L_E/L_P`, protocols `1-4`, and each participant separately; each folder includes a plot-data CSV and PNG manifest.


In [ ]:
import runpy
from pathlib import Path

script_path = find_repo_root() / "analysis" / "summery" / "generate_finger_annotated_summary_images.py"
runpy.run_path(str(script_path), run_name="__main__")